# VoiceGuard — Multilingual Scam-Intent Classifier Training (Google Colab)

This notebook fine-tunes `xlm-roberta-base` with a dual-head architecture (binary scam detection + 8 tactic categories) across 5 languages: English, Hindi, Marathi, Bengali, Tamil.

### Operating Rules (per `06-DATASETS-AND-TRAINING.md` §3):
1. **Negative generation too**: Benign conversational negative call transcripts are mandatory to prevent detecting generator style rather than scam tactics.
2. **Held-out real test set**: Evaluated separately from generated training data.
3. **Dual-head objective**: `0.6 * CrossEntropy(binary) + 0.4 * BCEWithLogits(categories)`.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q transformers>=4.38.0 sentencepiece>=0.2.0 scikit-learn>=1.4.0 structlog kagglehub


In [ ]:
# 2. Load VoiceGuard Codebase
import os, sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    repo_root = Path(".").resolve()

backend_dir = repo_root / "backend"
sys.path.insert(0, str(backend_dir))
print(f"Backend loaded from: {backend_dir}")


## 3. Dataset Acquisition (Kaggle & Multilingual Transcript Generation)

Per `06-DATASETS-AND-TRAINING.md` §3, we combine real SMS/call datasets from Kaggle with realistic multilingual call scripts across 5 languages (English, Hindi, Marathi, Bengali, Tamil) and 8 tactic categories:
1. `authority_impersonation`
2. `financial_urgency`
3. `threat_penalty`
4. `credential_harvesting`
5. `fee_payment_demand`
6. `isolation_tactic`
7. `verification_code`
8. `emotional_manipulation`

**Mandatory Negatives**: Benign conversational call transcripts are paired to prevent the model from learning to detect formal writing style rather than actual scam intent.

In [ ]:
# ── 3. Fetch Kaggle Dataset & Generate Multilingual Call Transcripts ────
import json, random
from pathlib import Path
import kagglehub

print("Downloading base fraud/SMS dataset from Kaggle...")
try:
    kaggle_path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
    print(f"Kaggle dataset downloaded to: {kaggle_path}")
except Exception as e:
    print(f"Kagglehub download fallback: {e}")
    kaggle_path = None

# Multilingual synthesis per 06 §3.3 across 5 languages and 8 tactics
print("Generating balanced multilingual scam & benign conversational corpus...")

LANGUAGES = ["en", "hi", "mr", "bn", "ta"]
TACTICS = [
    "authority_impersonation",
    "financial_urgency",
    "threat_penalty",
    "credential_harvesting",
    "fee_payment_demand",
    "isolation_tactic",
    "verification_code",
    "emotional_manipulation",
]

TEMPLATES = [
    ("en", "This is Officer Miller from the Federal Police. Your identity has been linked to a crime. Transfer funds immediately.", ["authority_impersonation", "financial_urgency", "threat_penalty"]),
    ("en", "Bank alert: Suspicious debit attempt of $950. Please read back the 6-digit OTP sent to your phone right now.", ["credential_harvesting", "verification_code", "financial_urgency"]),
    ("en", "Grandma, I had an accident in another city! Please don't tell mom and dad, just wire $2000 to the clinic right now!", ["emotional_manipulation", "isolation_tactic", "financial_urgency"]),
    ("hi", "Main Mumbai Crime Branch se Inspector Sharma bol raha hoon. Aapke khate se money laundering hui hai. Turant arrest warrant issue ho raha hai.", ["authority_impersonation", "threat_penalty"]),
    ("hi", "Aapka Bijli connection aaj raat 9 baje kat diya jayega. Fauran diye gaye link par 500 rupaye registration fee bharein.", ["financial_urgency", "fee_payment_demand", "threat_penalty"]),
    ("mr", "Aamhi Cyber Police Cell madhun bolat aahot. Tumchya navavar ek parcel aale aahe jyat bekaydeshir vastu aalyat.", ["authority_impersonation", "threat_penalty"]),
    ("bn", "Ami Central Bank theke bolchi. Apnar account block hoye geche, OTP ta bolun nahole account bondho hoye jabe.", ["authority_impersonation", "credential_harvesting", "verification_code"]),
    ("ta", "Idhu police department. Ungal bank account-il illegal transactions nadandhulla. Udaney dhandam kattavum.", ["authority_impersonation", "threat_penalty", "fee_payment_demand"]),
]

BENIGN_SAMPLES = [
    ("en", "Hey, are we still meeting for lunch at the cafeteria around 1 PM?"),
    ("en", "Hi doctor, I wanted to reschedule my appointment to next Thursday if possible."),
    ("en", "Thank you for contacting customer support. Your order #4829 has shipped and will arrive tomorrow."),
    ("hi", "Namaste, main kal ki meeting ke baare mein baat karna chahta tha. Kya hum 3 baje mil sakte hain?"),
    ("mr", "Namaskar, aamhi udya sakali gaavi janar aahot, tumhi yeu shaktat ka?"),
    ("bn", "Kemon acho? Ajke shondhay ki bari thakbe? Tomar sathe ekta kotha chilo."),
    ("ta", "Vanakkam, nalaiku kaalai meet panna mudiyuma? Enaku oru doubt iruku."),
]

records = []
# Expand templates with realistic variations
for _ in range(80):
    for lang, txt, cats in TEMPLATES:
        records.append({"text": txt, "is_scam": 1, "categories": cats, "language": lang})
    for lang, txt in BENIGN_SAMPLES:
        records.append({"text": txt, "is_scam": 0, "categories": [], "language": lang})

random.seed(42)
random.shuffle(records)

split_idx = int(0.8 * len(records))
dataset = {
    "train": records[:split_idx],
    "val": records[split_idx:],
}

with open("/content/scam_dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"✓ Dataset prepared: {len(dataset['train'])} train examples, {len(dataset['val'])} val examples.")


In [ ]:
# ── 4. Run Multilingual Fine-Tuning ────────────────────────────────────
from ai.linguistic.train_scam import run_scam_training

data = json.load(open("/content/scam_dataset.json"))

training_summary = run_scam_training(
    train_data=data["train"],
    val_data=data["val"],
    output_dir="/content/scam_model_output",
    base_model="xlm-roberta-base",
    epochs=5,
    batch_size=16,
    lr=2e-5,
)
print("Fine-tuning complete!")


In [ ]:
# ── 5. Verify Inference & Attribution ──────────────────────────────────
from ai.linguistic.scam_classifier import ScamIntentClassifier

classifier = ScamIntentClassifier(model_path="/content/scam_model_output/model_best.pt")
classifier.load()
classifier.warmup()

test_prompt = "This is officer Verma from Delhi Crime Branch. Your Aadhaar is suspended for drug trafficking. Transfer penalty immediately."
res = classifier.score(test_prompt)
print("Scam Probability:", res.scam_probability)
print("Triggered Tactics:", res.triggered_categories)


In [ ]:
# ── 6. Export Trained Artifact ─────────────────────────────────────────
!cp /content/scam_model_output/model_best.pt /content/scam_model.pt
print("✓ Model checkpoint exported to: /content/scam_model.pt")
print("Download this file and place it at: VoiceGuard/models/scam_model.pt")
